In [1]:
import re
import pandas as pd
import numpy as np
import gradio as gr
from catboost import CatBoostRegressor

def X_test_filter(X_train):
    X_train = X_train.replace("-", np.nan)
    X_train["Бренд"] = X_train["Бренд"].fillna(X_train["Бренд"].mode()[0])

    X_train["Модель"] = X_train["Модель"].fillna(X_train["Модель"].mode()[0])

    X_train["Тип машины"] = X_train["Тип машины"].fillna(X_train["Тип машины"].mode()[0])

    X_train["КПП"] = X_train["КПП"].fillna(X_train["КПП"].mode()[0])

    X_train["Двигатель"] = (X_train["Двигатель"].str.split().str[-2])
    X_train["Двигатель"] = pd.to_numeric(X_train["Двигатель"],errors="coerce")
    X_train["Двигатель"] = X_train["Двигатель"].fillna(X_train["Двигатель"].mode()[0])
    X_train["Двигатель"] = X_train["Двигатель"].astype(float)

    X_train["Топливо"] = X_train["Топливо"].fillna(X_train["Топливо"].mode()[0])

    X_train["Расход"] = (X_train["Расход"].str.split().str[0])
    X_train["Расход"] = pd.to_numeric(X_train["Расход"],errors="coerce")
    X_train["Расход"] = X_train["Расход"].fillna(X_train["Расход"].median())
    X_train["Расход"] = X_train["Расход"].astype(float)

    mask_swap = X_train["Пробег"].astype(str).str.contains(r"-\s*/\s*-", na=False)
    X_train.loc[mask_swap, ["Пробег", "Цвет"]] = X_train.loc[mask_swap, ["Цвет", "Пробег"]].values
    def extract_mileage(x):
        nums = re.findall(r"\d[\d,]*", str(x))
        if not nums:
            return None
        return int(nums[-1].replace(",", ""))
    X_train["Пробег"] = X_train["Пробег"].apply(extract_mileage)
    X_train["Пробег"] = X_train["Пробег"].fillna(X_train["Пробег"].median())
    X_train["Пробег"] = X_train["Пробег"].astype(float)

    X_train["Локация"] = (X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False)
        .fillna(X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False).mode()[0]))

    X_train["Количество цилиндров"] = (X_train["Количество цилиндров"].str.split().str[0])
    X_train["Количество цилиндров"] = X_train["Количество цилиндров"].fillna(X_train["Количество цилиндров"].mode()[0])
    X_train["Количество цилиндров"] = X_train["Количество цилиндров"].astype(float)

    X_train["Тип кузова"] = (X_train["Тип кузова"].str.split().str[0])
    X_train["Тип кузова"] = X_train["Тип кузова"].fillna(X_train["Тип кузова"].mode()[0])
    X_train["Тип кузова"] = X_train["Тип кузова"].astype(str)

    X_train["Двери"] = (X_train["Двери"].str.split().str[0])
    X_train["Двери"] = pd.to_numeric(X_train["Двери"],errors="coerce")
    X_train["Двери"] = X_train["Двери"].fillna(X_train["Двери"].mode()[0])
    X_train["Двери"] = X_train["Двери"].astype(float)

    X_train["Количество кресел"] = (X_train["Количество кресел"].str.split().str[0])
    X_train["Количество кресел"] = pd.to_numeric(X_train["Количество кресел"],errors="coerce")
    X_train["Количество кресел"] = X_train["Количество кресел"].fillna(X_train["Количество кресел"].mode()[0])
    X_train["Количество кресел"] = X_train["Количество кресел"].astype(float)

    X_train["Оценка эксперта"] = X_train["Оценка эксперта"].astype(float)

    X_train["Количество владельцев"] = X_train["Количество владельцев"].astype(float)
    return X_train

X_train = pd.read_csv("dataset_full/X_train.csv")
X_train = X_test_filter(X_train)
model = CatBoostRegressor()
model.load_model("catboost_model.cbm")

features1 = ["Год выпуска","Двигатель","Расход","Пробег","Бренд","Исползование","КПП","Привод","Топливо","Локация","Тип кузова",]

def predict_price(year,engine,fuel_consumption,mileage,brand,
        usage,gearbox,drive,fuel,location,body):

    data = pd.DataFrame([{
        "Год выпуска": year,
        "Двигатель": engine,
        "Расход": fuel_consumption,
        "Пробег": mileage,

        "Бренд": brand,
        "Исползование": usage,
        "КПП": gearbox,
        "Привод": drive,
        "Топливо": fuel,
        "Локация": location,
        "Тип кузова": body
    }])

    data1 = data[features1]

    prediction = model.predict(data1)[0]

    return f"{prediction:.0f}"

In [2]:
def predict_csv(file):

    df = pd.read_csv(file.name)
    df = X_test_filter(df)
    X = df[features1]

    predictions = model.predict(X)
    result = pd.DataFrame({
        "Цена": predictions.astype(int)
    })
    output_file = "submission_gradio.csv"
    result.to_csv(output_file, index=False)


    return output_file

In [3]:
inputs = [
    gr.Number(value=2020, label="Год выпуска"),
    gr.Number(value=2.0, label="Объем двигателя (л)"),
    gr.Number(value=8.0, label="Расход топлива (л/100км)"),
    gr.Number(value=50000, label="Пробег (км)"),
    gr.Dropdown(
        choices=sorted(X_train["Бренд"].dropna().unique().tolist()),
        label="Бренд"
    ),
    gr.Dropdown(
        choices=sorted(X_train["Исползование"].dropna().unique().tolist()),
        label="Использование"
    ),
    gr.Dropdown(
        choices=sorted(X_train["КПП"].dropna().unique().tolist()),
        label="КПП"
    ),
    gr.Dropdown(
        choices=sorted(X_train["Привод"].dropna().unique().tolist()),
        label="Привод"
    ),
    gr.Dropdown(
        choices=sorted(X_train["Топливо"].dropna().unique().tolist()),
        label="Топливо"
    ),
    gr.Dropdown(
        choices=sorted(X_train["Локация"].dropna().unique().tolist()),
        label="Локация"
    ),
    gr.Dropdown(
        choices=sorted(X_train["Тип кузова"].dropna().unique().tolist()),
        label="Тип кузова"
    ),
]
demo = gr.Interface(
    fn=predict_price,
    inputs=inputs,
    outputs="text",
    title="Оценка стоимости автомобиля",
)

csv_demo = gr.Interface(
    fn=predict_csv,
    inputs=gr.File(file_types=[".csv"]),outputs=gr.File(),
    title="Оценка автомобилей из CSV",
)

app = gr.TabbedInterface([demo,csv_demo],["Один автомобиль","CSV"])

app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
